In [1]:
# make_maps_and_diags.py  (cached, paper-ready)
# Genera: fig-deb-heatmap.png, fig-servc-access.png, fig-rpe-diversity.png
# Dipendenze: numpy, pandas, geopandas, shapely, networkx, matplotlib

import os, pickle, heapq, json
from pathlib import Path
from collections import defaultdict, deque

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt

from shapely.geometry import LineString, Point, box
from shapely.ops import unary_union, linemerge
import matplotlib.colors as mcolors
from matplotlib.ticker import LogLocator, LogFormatterMathtext, NullFormatter
from matplotlib.lines import Line2D

# ----------------------------
# Config (puoi cambiare QUI)
# ----------------------------
OUTPUT_DIR  = Path(".")
PICKLE_DIR  = Path("ulmm_pickles")
CACHE_DIR   = Path("cache"); CACHE_DIR.mkdir(parents=True, exist_ok=True)

CITY        = "New York City, New York, USA"
VEHICLE     = "van"            # "van" | "cargo_bike"
ORIENTATION = "AD"             # "AD" (Access->Demand) | "DA" (Demand->Access)
KAPPA       = 1.0              # penalità prox in ServC
RPE_K       = 25               # K-shortest per RPE
RPE_ETA     = 1.0              # temperatura Gibbs
SEED        = 7

SAVE_SVG    = False            # opzionale: esporta anche .svg

# stile figure consistente e leggibile
plt.rcParams.update({
    "figure.dpi": 180,
    "savefig.dpi": 300,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
})

# ----------------------------
# Helpers: slug & cache
# ----------------------------
def slugify(s: str) -> str:
    import re
    return re.sub(r"[^a-z0-9]+", "-", str(s).lower()).strip("-")

def ulmm_path_for_city(city_name: str) -> Path:
    return PICKLE_DIR / f"ulmm_{slugify(city_name)}.pkl"

def ulmm_fingerprint(ulmm_pkl_path: Path) -> str:
    try:
        st = ulmm_pkl_path.stat()
        return f"{st.st_mtime_ns}-{st.st_size}"
    except FileNotFoundError:
        return "missing"

def cache_file(prefix: str, **kwargs) -> Path:
    key = "__".join(f"{k}-{slugify(v)}" for k, v in sorted(kwargs.items()))
    return CACHE_DIR / f"{prefix}__{key}.pkl"

def cache_save(path: Path, data, meta: dict):
    payload = {"__meta__": meta, "__data__": data}
    with open(path, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)

def cache_load(path: Path, expected_meta: dict):
    if not path.exists():
        return None
    try:
        with open(path, "rb") as f:
            payload = pickle.load(f)
        meta = payload.get("__meta__", {})
        if meta == expected_meta:
            return payload.get("__data__")
        return None
    except Exception:
        return None

# ----------------------------
# Load ULMM
# ----------------------------
def load_ulmm(city_name: str) -> dict:
    p = ulmm_path_for_city(city_name)
    with open(p, "rb") as f:
        return pickle.load(f)

def to_local_crs(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    try:
        return gdf.to_crs(gdf.estimate_utm_crs())
    except Exception:
        return gdf

# ----------------------------
# Grafo collassato (MultiDiGraph → DiGraph) [CACHE]
# ----------------------------
def collapsed_graph(ulmm: dict, veh: str, force_recompute: bool = False):
    """Collassa MultiDiGraph scegliendo per (u,v) l'arco con peso minimo c_eff_<veh>.
       Restituisce (G, edge_geom, node_xy)."""
    city = ulmm["city"]
    meta = {
        "fingerprint": ulmm_fingerprint(ulmm_path_for_city(city)),
        "city": city, "veh": veh, "ver": "v1",
    }
    cpath = cache_file("collapsed_graph", city=city, veh=veh, fp=meta["fingerprint"])
    if not force_recompute:
        cached = cache_load(cpath, meta)
        if cached is not None:
            return cached

    Gm = ulmm["graph"]; weight_attr = f"c_eff_{veh}"
    G = nx.DiGraph()
    node_xy = {}
    for n, d in Gm.nodes(data=True):
        G.add_node(n)
        node_xy[n] = (d.get("x"), d.get("y"))

    edge_geom = {}
    for u, v, k, d in Gm.edges(keys=True, data=True):
        w = float(d.get(weight_attr, np.inf))
        if not np.isfinite(w):
            continue
        if (u, v) not in G or w < G[u][v]["weight"]:
            G.add_edge(u, v, weight=w)
            geom = d.get("geometry")
            if geom is None:
                x1, y1 = node_xy[u]; x2, y2 = node_xy[v]
                geom = LineString([(x1, y1), (x2, y2)])
            edge_geom[(u, v)] = geom

    out = (G, edge_geom, node_xy)
    cache_save(cpath, out, meta)
    return out

# ----------------------------
# Curb proximity media
# ----------------------------
def mean_phi_prox(Gm: nx.MultiDiGraph, node: int, veh: str, hops: int = 1) -> float:
    phis = []
    if hops <= 1:
        for _, _, _, d in Gm.edges(node, keys=True, data=True):
            v = d.get(f"phi_{veh}");  phis.append(float(v)) if v is not None else None
        for _, _, _, d in Gm.in_edges(node, keys=True, data=True):
            v = d.get(f"phi_{veh}");  phis.append(float(v)) if v is not None else None
    else:
        q = deque([(node,0)]); seen = {node}; nodes = {node}
        while q:
            u, h = q.popleft()
            if h == hops: continue
            for w in set(list(Gm.successors(u)) + list(Gm.predecessors(u))):
                if w not in seen:
                    seen.add(w); nodes.add(w); q.append((w, h+1))
        for u in nodes:
            for _, _, _, d in Gm.edges(u, keys=True, data=True):
                v = d.get(f"phi_{veh}");  phis.append(float(v)) if v is not None else None
    return float(np.mean(phis)) if phis else 0.0

# ----------------------------
# ServC [CACHE]
# ----------------------------
def compute_servc(ulmm: dict, veh: str, kappa: float = 0.0, force_recompute: bool = False) -> pd.DataFrame:
    city = ulmm["city"]
    meta = {
        "fingerprint": ulmm_fingerprint(ulmm_path_for_city(city)),
        "city": city, "veh": veh, "kappa": float(kappa), "ver": "v2",
    }
    cpath = cache_file("servc", city=city, veh=veh, kappa=kappa, fp=meta["fingerprint"])
    if not force_recompute:
        cached = cache_load(cpath, meta)
        if cached is not None:
            return cached

    G, _, _ = collapsed_graph(ulmm, veh)
    Gm = ulmm["graph"]
    demand, access = ulmm["demand"], ulmm["access"]

    d_nodes = demand["i_node"].astype(int).tolist()
    a_nodes = access["i_node"].astype(int).tolist()
    w_d     = demand["w"].astype(float).to_numpy()
    d_w = {int(n): float(w) for n, w in zip(d_nodes, w_d)}

    a_phi = {int(a): mean_phi_prox(Gm, int(a), veh) for a in a_nodes}

    # AD: access -> demand
    servc_AD = {}
    for a in a_nodes:
        dist = nx.single_source_dijkstra_path_length(G, int(a), weight="weight")
        val = 0.0
        for dn in d_nodes:
            t = dist.get(int(dn), np.inf)
            if np.isfinite(t):
                val += d_w[int(dn)] / (t + kappa*a_phi[int(a)] + 1e-9)
        servc_AD[int(a)] = val

    # DA: demand -> access (grafo invertito)
    G_rev = G.reverse(copy=False)
    servc_DA = {}
    for a in a_nodes:
        dist = nx.single_source_dijkstra_path_length(G_rev, int(a), weight="weight")
        val = 0.0
        for dn in d_nodes:
            t = dist.get(int(dn), np.inf)
            if np.isfinite(t):
                val += d_w[int(dn)] / (t + kappa*a_phi[int(a)] + 1e-9)
        servc_DA[int(a)] = val

    df = pd.DataFrame({
        "a_node": [int(n) for n in a_nodes],
        "a_id": access["a_id"].tolist(),
        "lat":   access["lat"].tolist(),
        "lon":   access["lon"].tolist(),
        "phi_prox": [a_phi[int(n)] for n in a_nodes],
        "servc_AD": [servc_AD[int(n)] for n in a_nodes],
        "servc_DA": [servc_DA[int(n)] for n in a_nodes],
    })
    cache_save(cpath, df, meta)
    return df

# ----------------------------
# DEB (Brandes pair-weighted) [CACHE]
# ----------------------------
def deb_pairweighted(ulmm: dict, veh: str, orientation: str = "AD", force_recompute: bool = False):
    city = ulmm["city"]
    meta = {
        "fingerprint": ulmm_fingerprint(ulmm_path_for_city(city)),
        "city": city, "veh": veh, "ori": orientation, "ver": "v2",
    }
    cpath = cache_file("deb_pairweighted", city=city, veh=veh, ori=orientation, fp=meta["fingerprint"])
    if not force_recompute:
        cached = cache_load(cpath, meta)
        if cached is not None:
            return cached

    G, edge_geom, node_xy = collapsed_graph(ulmm, veh)
    demand, access = ulmm["demand"], ulmm["access"]
    d_nodes = [int(n) for n in demand["i_node"].tolist()]
    a_nodes = [int(n) for n in access["i_node"].tolist()]
    w_d     = demand["w"].astype(float).to_numpy()

    if orientation.upper() == "AD":
        sources = a_nodes; targets = set(d_nodes)
        t_weight = {int(n): float(w) for n, w in zip(d_nodes, w_d)}
    else:
        sources = d_nodes; targets = set(a_nodes)
        t_weight = {int(n): 1.0 for n in a_nodes}

    edges = {(u, v): 0.0 for u, v in G.edges()}
    for s in sources:
        S, P = [], defaultdict(list)
        sigma = defaultdict(float)
        dist  = defaultdict(lambda: np.inf)
        sigma[s] = 1.0; dist[s] = 0.0
        Q = [(0.0, s)]
        while Q:
            dv, v = heapq.heappop(Q)
            if dv > dist[v] + 1e-12: continue
            S.append(v)
            for w in G.successors(v):
                vw = G[v][w]["weight"]
                dth = dist[v] + vw
                if dth + 1e-12 < dist[w]:
                    dist[w] = dth; heapq.heappush(Q, (dist[w], w))
                    sigma[w] = sigma[v]; P[w] = [v]
                elif abs(dth - dist[w]) <= 1e-12:
                    sigma[w] += sigma[v]; P[w].append(v)

        delta = defaultdict(float); b = defaultdict(float)
        for t in targets:
            if np.isfinite(dist[t]): b[t] = t_weight.get(t, 0.0)

        while S:
            w = S.pop()
            coeff_w = b[w] + delta[w]
            for v in P[w]:
                if sigma[w] > 0:
                    c = (sigma[v]/sigma[w]) * coeff_w
                    edges[(v, w)] += c
                    delta[v] += c

    vals = np.array(list(edges.values()), float)
    tot  = float(vals.sum()) if vals.sum() > 0 else 1.0
    deb_norm = {e: (val/tot) for e, val in edges.items()}

    geoms, vlist = [], []
    for (u, v), val in deb_norm.items():
        geom = edge_geom.get((u, v))
        if geom is None:
            x1, y1 = node_xy[u]; x2, y2 = node_xy[v]
            geom = LineString([(x1, y1), (x2, y2)])
        geoms.append(geom); vlist.append(val)
    g_edges = gpd.GeoDataFrame({"deb_norm": vlist}, geometry=geoms, crs="EPSG:4326")

    out = (deb_norm, g_edges)
    cache_save(cpath, out, meta)
    return out

# ----------------------------
# RPE (K-shortest verso qualunque access) [CACHE]
# ----------------------------
def k_shortest_to_any_access(G: nx.DiGraph, src: int, access_nodes: list, K: int, weight_key: str = "weight"):
    H = G.copy(); SINK = "__SINK__"; H.add_node(SINK)
    for a in set(access_nodes):
        if a in H:
            H.add_edge(a, SINK, **{weight_key: 0.0})
    try:
        gen = nx.shortest_simple_paths(H, src, SINK, weight=weight_key)
        paths = []
        for _, p in zip(range(K), gen):
            if len(p) >= 2 and p[-1] == SINK:
                paths.append(p[:-1])
        return paths
    except nx.NetworkXNoPath:
        return []

def rpe_for_all_demands(ulmm: dict, veh: str, K: int = 15, eta: float = 1.0,
                        orientation: str = "AD", force_recompute: bool = False):
    city = ulmm["city"]
    meta = {
        "fingerprint": ulmm_fingerprint(ulmm_path_for_city(city)),
        "city": city, "veh": veh, "K": int(K), "eta": float(eta),
        "ori": orientation, "ver": "v2",
    }
    cpath = cache_file("rpe", city=city, veh=veh, K=K, eta=eta, ori=orientation, fp=meta["fingerprint"])
    if not force_recompute:
        cached = cache_load(cpath, meta)
        if cached is not None:
            return cached

    G, edge_geom, node_xy = collapsed_graph(ulmm, veh)
    demand, access = ulmm["demand"], ulmm["access"]
    d_nodes = [int(n) for n in demand["i_node"].tolist()]
    a_nodes = [int(n) for n in access["i_node"].tolist()]

    G_use = G; sources = d_nodes; targets = a_nodes
    rpe_vals, paths_examples = {}, {}
    for d in sources:
        kpaths = k_shortest_to_any_access(G_use, d, targets, K, weight_key="weight")
        if not kpaths:
            rpe_vals[d] = 0.0; paths_examples[d] = []; continue
        lens = []
        for p in kpaths:
            L = 0.0
            for u, v in zip(p[:-1], p[1:]): L += G_use[u][v]["weight"]
            lens.append(L)
        lens  = np.asarray(lens, float)
        probs = np.exp(-eta * lens); Z = probs.sum()
        probs = probs / (Z if Z > 0 else 1.0)
        eps = 1e-12
        H = -float(np.sum(probs * np.log(np.clip(probs, eps, 1.0))))
        rpe_vals[d] = H; paths_examples[d] = kpaths

    out = (rpe_vals, paths_examples, G, edge_geom, node_xy)
    cache_save(cpath, out, meta)
    return out

# ----------------------------
# Utilities per RPE plotting
# ----------------------------
def _paths_lengths_probs(G, paths, eta=1.0):
    if not paths: return [], [], []
    lens = []
    for p in paths:
        L = 0.0
        for u, v in zip(p[:-1], p[1:]): L += G[u][v]["weight"]
        lens.append(L)
    lens = np.asarray(lens, float)
    w = np.exp(-eta * lens); Z = w.sum()
    probs = w / (Z if Z > 0 else 1.0)
    return lens, probs, np.argsort(lens)

def _to_lines_gdf(paths, edge_geom, node_xy, probs=None, crs="EPSG:4326"):
    geoms, pvals = [], []
    for i, p in enumerate(paths):
        segs = []
        for u, v in zip(p[:-1], p[1:]):
            geom = edge_geom.get((u, v))
            if geom is None:
                x1, y1 = node_xy[u]; x2, y2 = node_xy[v]
                geom = LineString([(x1, y1), (x2, y2)])
            segs.append(geom)
        try:
            geom_u = linemerge(unary_union(segs))
        except Exception:
            geom_u = LineString([pt for g in segs for pt in g.coords])
        geoms.append(geom_u)
        pvals.append(probs[i] if probs is not None else 1.0)
    return gpd.GeoDataFrame({"prob": pvals}, geometry=geoms, crs=crs)

def _clip_bbox(g_lines, pad_ratio=0.08):
    b = g_lines.total_bounds
    if not np.isfinite(b).all(): return None
    dx, dy = (b[2]-b[0]), (b[3]-b[1])
    return box(b[0]-dx*pad_ratio, b[1]-dy*pad_ratio, b[2]+dx*pad_ratio, b[3]+dy*pad_ratio)

def _iter_line_xy(geom):
    if geom is None or geom.is_empty: return
    if hasattr(geom, "geoms"):
        for g in geom.geoms: yield from _iter_line_xy(g); return
    if hasattr(geom, "exterior"):
        try: yield geom.exterior.xy; return
        except Exception: pass
    if hasattr(geom, "xy"):
        try: yield geom.xy; return
        except Exception: pass
    try:
        arr = np.asarray(geom.coords)
        if arr.ndim == 2 and arr.shape[1] >= 2: yield arr[:,0], arr[:,1]
    except Exception:
        return

# ----------------------------
# PLOTTING
# ----------------------------
def plot_deb_heatmap(ulmm, veh, orientation, out_path):
    deb_norm, g_edges = deb_pairweighted(ulmm, veh, orientation)
    g_edges = to_local_crs(g_edges)

    vals = g_edges["deb_norm"].values
    q90, q99 = np.quantile(vals, [0.90, 0.99])
    vmax = float(np.nanmax(vals)) or 1.0
    cmap = plt.cm.plasma
    norm = mcolors.LogNorm(vmin=max(q90, 1e-9), vmax=vmax)

    fig = plt.figure(figsize=(12.5, 8))
    ax = fig.add_axes([0.05, 0.05, 0.73, 0.90])

    # base network
    g_edges.plot(ax=ax, color="#dcdcdc", linewidth=0.25, alpha=0.55, zorder=1)

    # 90–100% colorati
    sel90 = g_edges[vals >= q90].copy(); h = None
    if not sel90.empty:
        h = sel90.plot(ax=ax, column="deb_norm", cmap=cmap, norm=norm,
                       linewidth=0.9, alpha=0.95, zorder=2)

    # top-1% evidenziato con stessa cmap ma più spesso
    sel99 = g_edges[vals >= q99].copy()
    if not sel99.empty:
        colors_top = cmap(norm(sel99["deb_norm"].values))
        sel99.plot(ax=ax, color=colors_top, linewidth=1.9, alpha=0.98, zorder=3)

    ax.set_axis_off()

    # colorbar
    if h is not None:
        cax = fig.add_axes([0.955, 0.55, 0.02, 0.35])
        cb = fig.colorbar(h.collections[0], cax=cax)
        cb.set_label("normalized DEB share (q≥90%, log)")

    # Lorenz
    vals_pos = np.sort(vals[vals > 0])[::-1]
    if vals_pos.size == 0: vals_pos = np.array([1.0])
    axL = fig.add_axes([0.78, 0.12, 0.17, 0.31])
    cum = np.cumsum(vals_pos)/vals_pos.sum(); cum = np.insert(cum, 0, 0.0)
    x2 = np.linspace(0, 1, len(cum), endpoint=True)
    axL.plot(x2, cum, lw=1.4); axL.plot([0,1],[0,1],"--",lw=1.0,color="#777")
    axL.set_title("Lorenz"); axL.set_xlabel("Edges share", labelpad=2)
    axL.set_ylabel("DEB share", labelpad=2); axL.tick_params(labelsize=8, pad=1)

    # Rank–size
    axR = fig.add_axes([0.78, 0.60, 0.17, 0.31])
    ranks = np.arange(1, len(vals_pos)+1)
    axR.plot(ranks, vals_pos, lw=1.2)
    axR.set_xscale("log"); axR.set_yscale("log")
    for axis in (axR.xaxis, axR.yaxis):
        axis.set_major_locator(LogLocator(base=10.0, numticks=4))
        axis.set_major_formatter(LogFormatterMathtext(base=10.0))
        axis.set_minor_formatter(NullFormatter())
    axR.set_title("Rank–size"); axR.set_xlabel("rank", labelpad=2); axR.set_ylabel("value", labelpad=2)
    axR.tick_params(labelsize=8, pad=1)

    # mini-legend
    handle = Line2D([0],[0], color="black", lw=2.0)
    ax.legend([handle], ["Top 1% thickness"], loc="lower left", frameon=True)

    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    if SAVE_SVG: fig.savefig(out_path.with_suffix(".svg"), bbox_inches="tight")
    plt.close(fig)

# --- helper per mettere la legenda fuori dal pannello, senza coprire la mappa
def _place_size_legend(fig, ax, color, all_values, scale_fn, title="ServC size"):
    # quantili p10/p50/p90 -> tre marker dimostrativi
    qv = np.quantile(all_values, [0.10, 0.50, 0.90])
    sizes = scale_fn(qv)

    handles = [
        plt.scatter([], [], s=sizes[0], color=color, alpha=0.80, edgecolors="white", linewidths=0.6),
        plt.scatter([], [], s=sizes[1], color=color, alpha=0.30, edgecolors=color, linewidths=0.8),
        plt.scatter([], [], s=sizes[2], color=color, alpha=0.85, edgecolors="white", linewidths=0.6),
    ]
    labels = ["P10", "P50", "P90"]

    # posizione: poco sotto il riquadro del pannello (in coordinate figura)
    pos = ax.get_position()
    leg = fig.legend(
        handles, labels, title=title,
        loc="lower left",
        bbox_to_anchor=(pos.x0 + 0.01, pos.y0 - 0.09),   # <— spostato fuori
        bbox_transform=fig.transFigure,
        frameon=True, borderpad=0.6, labelspacing=0.4, handletextpad=0.6,
        scatterpoints=1,
    )
    f = leg.get_frame()
    f.set_facecolor("white"); f.set_alpha(1.0)
    f.set_edgecolor("#cfcfcf"); f.set_linewidth(1.1)

def plot_servc_access(ulmm, veh, kappa, out_path):
    df = compute_servc(ulmm, veh, kappa)

    # ---- background curb (phi) su edges
    Gm = ulmm["graph"]
    geoms, phis = [], []
    for u, v, k, d in Gm.edges(keys=True, data=True):
        phi = d.get(f"phi_{veh}")
        if phi is None:
            continue
        if d.get("geometry") is not None:
            geoms.append(d["geometry"])
        else:
            x1, y1 = Gm.nodes[u]["x"], Gm.nodes[u]["y"]
            x2, y2 = Gm.nodes[v]["x"], Gm.nodes[v]["y"]
            geoms.append(LineString([(x1, y1), (x2, y2)]))
        phis.append(float(phi))
    g_edges = gpd.GeoDataFrame({"phi": phis}, geometry=geoms, crs="EPSG:4326")
    g_edges = to_local_crs(g_edges)

    # ---- access points
    g_acc = gpd.GeoDataFrame(
        df[["a_node", "servc_AD", "servc_DA", "phi_prox"]].copy(),
        geometry=[Point(lon, lat) for lon, lat in zip(df["lon"], df["lat"])],
        crs="EPSG:4326"
    )
    g_acc = to_local_crs(g_acc)

    # --- scaling condiviso
    sAD = g_acc["servc_AD"].values.astype(float)
    sDA = g_acc["servc_DA"].values.astype(float)
    s_all = np.concatenate([sAD, sDA])
    smin, smax = np.quantile(s_all, [0.05, 0.95])  # robust
    def scale_sizes(x, min_sz=18, max_sz=260):
        x = np.clip((x - smin) / (smax - smin + 1e-9), 0, 1)
        return min_sz + x * (max_sz - min_sz)
    sizes_DA = scale_sizes(sDA); sizes_AD = scale_sizes(sAD)

    # --- figure
    fig, axes = plt.subplots(1, 3, figsize=(18, 8))

    # basemap grigia
    for ax in axes:
        g_edges.plot(ax=ax, column="phi", cmap="Greys", linewidth=0.35, alpha=0.45)
        ax.set_axis_off()

    # pannello 1: DA
    ax = axes[0]
    top_mask_DA = g_acc["servc_DA"] >= np.quantile(g_acc["servc_DA"], 0.90)
    g_acc.plot(ax=ax, markersize=sizes_DA, color="#1f77b4", alpha=0.75,
               edgecolor="white", linewidth=0.4)
    g_acc[top_mask_DA].plot(ax=ax, markersize=sizes_DA[top_mask_DA]*1.05, facecolor="none",
                            edgecolor="#084c8d", linewidth=1.1, alpha=0.95)
    ax.set_title(f"ServC← (DA) — {ulmm['city']} [{veh}], κ={kappa}")

    # pannello 2: AD
    ax = axes[1]
    top_mask_AD = g_acc["servc_AD"] >= np.quantile(g_acc["servc_AD"], 0.90)
    g_acc.plot(ax=ax, markersize=sizes_AD, color="#2ca02c", alpha=0.75,
               edgecolor="white", linewidth=0.4)
    g_acc[top_mask_AD].plot(ax=ax, markersize=sizes_AD[top_mask_AD]*1.05, facecolor="none",
                            edgecolor="#1b6e1b", linewidth=1.1, alpha=0.95)
    ax.set_title(f"ServC→ (AD) — {ulmm['city']} [{veh}], κ={kappa}")

    # pannello 3: AD vs DA (log-ratio), size = mean ServC
    ax = axes[2]
    log_ratio = np.log2((sAD + 1e-12) / (sDA + 1e-12))
    avg_sz = scale_sizes(0.5 * (sAD + sDA))
    vmax = np.quantile(np.abs(log_ratio), 0.98)
    norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)
    g_acc.assign(L=log_ratio).plot(
        ax=ax, column="L", cmap="coolwarm", norm=norm,
        markersize=avg_sz, alpha=0.85, edgecolor="white", linewidth=0.4
    )
    ax.set_title("AD vs DA (log₂ ratio) — size = mean ServC")

    # >>> legende spostate FUORI dai pannelli 1 e 2
    _place_size_legend(fig, axes[0], "#1f77b4", s_all, scale_sizes, title="ServC size")
    _place_size_legend(fig, axes[1], "#2ca02c", s_all, scale_sizes, title="ServC size")

    # colorbar del pannello 3
    sm = plt.cm.ScalarMappable(cmap="coolwarm", norm=norm); sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes[2], fraction=0.035, pad=0.02)
    cbar.set_label("log₂(ServC→ / ServC←)")

    fig.tight_layout()
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

def plot_rpe_distribution(ulmm, veh, K, eta, orientation, out_path, seed=7):
    rng = np.random.default_rng(seed)
    rpe_vals, paths_by_d, G, edge_geom, node_xy = rpe_for_all_demands(
        ulmm, veh, K, eta, orientation
    )

    # ---- H/logK
    items = list(rpe_vals.items())
    if not items:
        Hnorm = np.array([0.0])
    else:
        Hnorm = np.array([v for _, v in items], float) / max(np.log(max(K,1)), 1e-9)

    # esempi low/high
    i_low, i_high = int(np.argmin(Hnorm)), int(np.argmax(Hnorm))
    d_low,  h_low  = items[i_low][0],  float(Hnorm[i_low])
    d_high, h_high = items[i_high][0], float(Hnorm[i_high])

    # K-corridors e pesi Gibbs
    def _paths_lengths_probs(G, paths, eta=1.0):
        if not paths: return [], []
        lens = []
        for p in paths:
            L = 0.0
            for u, v in zip(p[:-1], p[1:]): L += G[u][v]["weight"]
            lens.append(L)
        lens = np.asarray(lens, float)
        w = np.exp(-eta * lens); Z = w.sum()
        return (w / (Z if Z > 0 else 1.0)).tolist()

    k_low   = paths_by_d.get(d_low,  [])[:K]
    k_high  = paths_by_d.get(d_high, [])[:K]
    p_low   = _paths_lengths_probs(G, k_low,  eta)
    p_high  = _paths_lengths_probs(G, k_high, eta)

    # GDF delle linee (una geometria per percorso, con colonna "prob")
    def _to_lines_gdf(paths, probs, crs="EPSG:4326"):
        geoms, pvals = [], []
        for i, p in enumerate(paths):
            segs = []
            for u, v in zip(p[:-1], p[1:]):
                g = edge_geom.get((u, v))
                if g is None:
                    x1, y1 = node_xy[u]; x2, y2 = node_xy[v]
                    g = LineString([(x1, y1), (x2, y2)])
                segs.append(g)
            try:
                g_u = linemerge(unary_union(segs))
            except Exception:
                g_u = LineString([pt for g in segs for pt in g.coords])
            geoms.append(g_u); pvals.append(float(probs[i]) if probs else 1.0)
        return gpd.GeoDataFrame({"prob": pvals}, geometry=geoms, crs=crs)

    g_low  = to_local_crs(_to_lines_gdf(k_low,  p_low))
    g_high = to_local_crs(_to_lines_gdf(k_high, p_high))

    # background locale
    bbox = None
    if not g_low.empty and not g_high.empty:
        bbox = _clip_bbox(gpd.GeoDataFrame(
            geometry=list(g_low.geometry) + list(g_high.geometry), crs=g_low.crs
        ))
    bg = None
    if bbox is not None:
        Gm = ulmm["graph"]; ge = []
        for u, v, k, d in Gm.edges(keys=True, data=True):
            g = d.get("geometry")
            if g is None:
                x1, y1 = Gm.nodes[u]["x"], Gm.nodes[u]["y"]
                x2, y2 = Gm.nodes[v]["x"], Gm.nodes[v]["y"]
                g = LineString([(x1, y1), (x2, y2)])
            ge.append(g)
        bg = to_local_crs(gpd.GeoDataFrame(geometry=ge, crs="EPSG:4326")).clip(bbox)

    # ===== layout allineato in altezza =====
    fig = plt.figure(figsize=(14, 6))
    gs = fig.add_gridspec(
        nrows=1, ncols=3, left=0.06, right=0.94, bottom=0.12, top=0.90,
        wspace=0.20, width_ratios=[1.25, 1.0, 1.0]
    )
    ax_hist = fig.add_subplot(gs[0, 0])
    ax_low  = fig.add_subplot(gs[0, 1])
    ax_high = fig.add_subplot(gs[0, 2])

    # --- istogramma
    ax_hist.hist(Hnorm, bins=30, alpha=0.9)
    ax_hist.axvline(0.0, color="#d62728", linestyle="--", linewidth=1.2, label="0 (single path)")
    ax_hist.axvline(1.0, color="#2ca02c", linestyle="--", linewidth=1.2, label="1 (≈ log K)")
    ax_hist.set_title(f"RPE distribution — {ulmm['city']} [{veh}] (K={K}, η={eta})")
    ax_hist.set_xlabel("RPE_normalized = H / log(K)")
    ax_hist.set_ylabel("# demand nodes")
    ax_hist.legend(loc="upper left")

    # --- normalizzazione colori condivisa (p5–p95) + linewidth coerente
    probs_all = np.array((p_low or []) + (p_high or []), float)
    if probs_all.size == 0:
        vmin, vmax = 0.0, 1.0
    else:
        vmin = float(np.quantile(probs_all, 0.05))
        vmax = float(np.quantile(probs_all, 0.95))
        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin >= vmax:
            vmin, vmax = float(probs_all.min()), float(probs_all.max())
            if vmin == vmax: vmin, vmax = 0.0, 1.0
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax, clip=True)
    cmap = plt.cm.viridis

    def _w(p):
        # spessore relativo sulla stessa finestra p5–p95 (visibile anche per p≈1/K)
        t = (float(p) - vmin) / (vmax - vmin + 1e-9)
        t = np.clip(t, 0.0, 1.0)
        return 0.8 + 3.2*t

    # --- Low
    if bg is not None: bg.plot(ax=ax_low, color="#e5e5e5", linewidth=0.3, alpha=0.8)
    if not g_low.empty:
        for _, row in g_low.iterrows():
            for x, y in _iter_line_xy(row.geometry):
                ax_low.plot(x, y, color=cmap(norm(row.prob)), linewidth=_w(row.prob))
    ax_low.set_axis_off()
    ax_low.set_title(f"Low RPE (H/logK={h_low:.2f})")

    # --- High
    if bg is not None: bg.plot(ax=ax_high, color="#e5e5e5", linewidth=0.3, alpha=0.8)
    if not g_high.empty:
        for _, row in g_high.iterrows():
            for x, y in _iter_line_xy(row.geometry):
                ax_high.plot(x, y, color=cmap(norm(row.prob)), linewidth=_w(row.prob))
    ax_high.set_axis_off()
    ax_high.set_title(f"High RPE (H/logK={h_high:.2f})")

    # --- colorbar della stessa altezza del pannello destro
    pos = ax_high.get_position()
    cax = fig.add_axes([pos.x1 + 0.012, pos.y0, 0.02, pos.height])
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    cb = fig.colorbar(sm, cax=cax)
    cb.set_label("Path probability (p5–p95)")

    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

# ----------------------------
# MAIN
# ----------------------------
ulmm = load_ulmm(CITY)
city_slug = slugify(CITY)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# Figure 1: DEB heatmap + inset
out1 = OUTPUT_DIR / "fig-deb-heatmap.png"
plot_deb_heatmap(ulmm, VEHICLE, ORIENTATION, out1)
print(f"[ok] {out1}")

[ok] fig-deb-heatmap.png


In [10]:
# Figure 2: ServC access map
out2 = OUTPUT_DIR / "fig-servc-access.png"
plot_servc_access(ulmm, VEHICLE, KAPPA, out2)
print(f"[ok] {out2}")

[ok] fig-servc-access.png


In [2]:
# Figure 3: RPE distribution + corridors
out3 = OUTPUT_DIR / "fig-rpe-diversity.png"
plot_rpe_distribution(ulmm, VEHICLE, RPE_K, RPE_ETA, ORIENTATION, out3, seed=SEED)
print(f"[ok] {out3}")

[ok] fig-rpe-diversity.png
